In [ ]:
import pickle
import random
from src.bee_classes import Word

# Load study lists created by main analysis notebook
with open('data/study_lists.pkl', 'rb') as f:
    study_categories = pickle.load(f)

print("📚 Study lists loaded!")
print(f"  Singleton 4-letter: {len(study_categories['singleton_4letter'])}")
print(f"  Multi 4-letter: {len(study_categories['multi_4letter'])}")
print(f"  Singleton 5+: {len(study_categories['singleton_5plus'])}")
print(f"  Multi 5+: {len(study_categories['multi_5plus'])}")

## Quiz Engine

In [ ]:
RED = '\033[91m'
BOLD = '\033[1m'
GREEN = '\033[92m'
END = '\033[0m'

def quiz_alphagram(alphagram, all_valid_words, focus_word=None, reveal_threshold=5):
    """
    Quiz for a single alphagram.
    
    Args:
        alphagram: The alphagram to quiz
        all_valid_words: List of all valid words for this alphagram
        focus_word: Optional - the word from your difficulty list (highlighted if found)
        reveal_threshold: Number of wrong guesses before revealing answers
    """
    found_words = []
    wrong_guesses = 0
    
    print(f"\n{BOLD}{RED}{'='*60}{END}")
    print(f"{BOLD}{RED}Alphagram: {alphagram}{END}")
    print(f"Total valid words: {len(all_valid_words)}")
    print(f"{BOLD}{RED}{'='*60}{END}\n")
    
    while len(found_words) < len(all_valid_words):
        remaining = [w for w in all_valid_words if w not in found_words]
        
        print(f"\n✅ Found ({len(found_words)}): {', '.join(sorted(found_words))}")
        print(f"❓ Remaining: {len(remaining)} word(s)")
        
        # Give hints after wrong guesses
        if wrong_guesses > 0 and remaining:
            hint_level = min(wrong_guesses, 3)
            hints = [f"{w[:hint_level]}{'.' * (len(w) - hint_level)}" for w in sorted(remaining)]
            print(f"💡 Hints: {', '.join(hints)}")
        
        guess_input = input(f"\nGuess word(s) for {BOLD}{alphagram}{END} (X=skip, XXX=quit): ").upper().strip()
        
        if guess_input == 'XXX':
            return 'QUIT'
        elif guess_input == 'X':
            print(f"\n🔍 Words you missed: {', '.join(sorted(remaining))}")
            return 'SKIP'
        
        # Split input to allow multiple words
        guesses = guess_input.split()
        
        for guess in guesses:
            if guess in all_valid_words:
                if guess in found_words:
                    print(f"⚠️  You already found {guess}!")
                else:
                    found_words.append(guess)
                    if guess == focus_word:
                        print(f"{GREEN}{BOLD}🎯 Excellent! You found the focus word: {guess}{END}")
                    else:
                        print(f"{GREEN}✓ Correct! {guess}{END}")
            else:
                # Check if it's a valid word with wrong alphagram
                try:
                    guess_alphagram = Word(guess).alphagram()
                    if guess_alphagram != alphagram:
                        print(f"❌ '{guess}' has alphagram {guess_alphagram}, not {alphagram}")
                    else:
                        print(f"❌ '{guess}' is not valid for this puzzle (check scrabble dict or past puzzles)")
                except ValueError:
                    # Word has too many/few letters or other validation issue
                    print(f"❌ '{guess}' is not a valid word format")
                wrong_guesses += 1
            
            if wrong_guesses >= reveal_threshold:
                print(f"\n🔍 Too many misses. Remaining words: {', '.join(sorted(remaining))}")
                return 'REVEALED'
    
    print(f"\n{GREEN}{BOLD}🎉 Perfect! You found all {len(all_valid_words)} words!{END}")
    return 'COMPLETE'

print("✅ Quiz engine loaded")

## Run Quiz by Category

Choose which category to quiz yourself on.

In [ ]:
def run_category_quiz(category_name, max_words=None, randomize=True):
    """
    Run quiz for a specific category.
    """
    words = study_categories[category_name].copy()
    
    if randomize:
        random.shuffle(words)
    
    if max_words:
        words = words[:max_words]
    
    print(f"\n{'='*70}")
    print(f"Starting quiz: {category_name.replace('_', ' ').title()}")
    print(f"Total words: {len(words)}")
    print(f"{'='*70}")
    
    stats = {'complete': 0, 'skip': 0, 'revealed': 0}
    
    for i, entry in enumerate(words, 1):
        print(f"\n\n📝 Word {i} of {len(words)}")
        result = quiz_alphagram(
            alphagram=entry['alphagram'],
            all_valid_words=entry['alphagram_words'],
            focus_word=entry['word']
        )
        
        if result == 'QUIT':
            print("\n👋 Quiz ended early")
            break
        elif result == 'COMPLETE':
            stats['complete'] += 1
        elif result == 'SKIP':
            stats['skip'] += 1
        elif result == 'REVEALED':
            stats['revealed'] += 1
    
    # Show stats
    print(f"\n\n{'='*70}")
    print("Quiz Complete! 📊 Statistics:")
    print(f"  ✅ Perfect: {stats['complete']}")
    print(f"  ⏭️  Skipped: {stats['skip']}")
    print(f"  🔍 Revealed: {stats['revealed']}")
    print(f"{'='*70}")

print("✅ Quiz runner loaded")
print("\nExample usage:")
print("  run_category_quiz('singleton_4letter', max_words=10)")
print("  run_category_quiz('multi_5plus', max_words=20, randomize=True)")

## Quick Start Examples

Uncomment one to run:

In [ ]:
# Quiz 10 random singleton 4-letter words
# run_category_quiz('singleton_4letter', max_words=10, randomize=False)

# Quiz 20 multi-word 5+ letter alphagrams
run_category_quiz('multi_5plus', max_words=10, randomize=False)

# Quiz ALL words in a category (be prepared!)
# run_category_quiz('singleton_5plus', randomize=True)